# Credit Score Soft Check Project
**Databricks Data Engineer Associate Demo**  
*by [Abhipray Lal]*

## Project Architecture
```mermaid
graph TD
    A[Faker Data] --> B[Bronze]
    B --> C[Silver] 
    C --> D[Gold]
    D --> E[Dashboard]
```

In [0]:
%pip install faker
from faker import Faker
fake = Faker()

Python interpreter will be restarted.
Python interpreter will be restarted.


In [0]:
# MAGIC %md
# MAGIC ### Project Setup
# MAGIC *Databricks Data Engineer Associate Demo/Credit Score Check_Abhipray Lal*

# COMMAND ----------
# Initialize Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Create databases (if they don't exist)
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# Verify setup
display(spark.sql("SHOW DATABASES"))

databaseName
bronze
default
gold
silver


In [0]:
# MAGIC %md
# MAGIC ### Generate User Data

# COMMAND ----------
from pyspark.sql.functions import lit
import random
from pyspark.sql.types import IntegerType, StringType

# 1. First force-clean any existing table
spark.sql("DROP TABLE IF EXISTS bronze.users")

# 2. Create sample data with explicit schema
users_data = [(i, f"User_{i}", random.randint(18, 70), 
              random.choice(["employed", "unemployed"]), 
              random.randint(20000, 150000)) 
             for i in range(100)]

# 3. Create DataFrame with enforced schema
users_df = spark.createDataFrame(
    users_data,
    schema="user_id INT, name STRING, age INT, employment_status STRING, income INT"
)

# 4. Save with schema overwrite
users_df.write \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("bronze.users")

# 5. Verify
print("Schema after write:")
users_df.printSchema()
display(spark.table("bronze.users").limit(99))

Schema after write:
root
 |-- user_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- income: integer (nullable = true)



user_id,name,age,employment_status,income
84,User_84,43,unemployed,32307
85,User_85,18,unemployed,130851
86,User_86,67,employed,106622
87,User_87,25,employed,21253
88,User_88,64,employed,98713
89,User_89,21,employed,117477
90,User_90,19,unemployed,72263
91,User_91,65,employed,128657
92,User_92,66,unemployed,105071
93,User_93,55,unemployed,134732


In [0]:
# MAGIC %md
# MAGIC ### Generating Valid Credit Card Data


# COMMAND ----------
from pyspark.sql.functions import lit, floor, concat
import random

# 1. Clean up existing table
spark.sql("DROP TABLE IF EXISTS bronze.credit_cards")

# 2. Get users (ensure we use the same DF as before)
users = spark.table("bronze.users").select("user_id")

# 3. Generate COMPLETE card data - NO NULLS
cards_df = users.withColumns({
    # Realistic card number (last 4 digits)
    "card_number": concat(
        lit("4111-1111-1111-"),
        (floor(rand() * 9000) + 1000).cast("int").cast("string")
    ),
    # Reasonable limits based on income
    "limit": (lit(5000) + (rand() * 15000)).cast("decimal(10,2)"),
    # Balance <= 80% of limit
    "balance": (col("limit") * rand() * 0.8).cast("decimal(10,2)"),
    # At least 12 on-time payments
    "on_time_payments": (lit(12) + (rand() * 48)).cast("int"),
    # 0-5 late payments
    "late_payments": (rand() * 5).cast("int")
})

# 4. Enforce data quality BEFORE save
assert cards_df.filter(col("card_number").isNull()).count() == 0
assert cards_df.filter(col("balance") > col("limit")).count() == 0

# 5. Save with validation
cards_df.write \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("bronze.credit_cards")

# 6. Verify
print("Data Quality Checks:")
print(f"Total cards: {cards_df.count()}")
print(f"Users without cards: {users.count() - cards_df.count()}")
display(cards_df.orderBy("user_id").limit(99))

Data Quality Checks:
Total cards: 100
Users without cards: 0


user_id,card_number,limit,balance,on_time_payments,late_payments
0,4111-1111-1111-3911,14838.27,1992.74,54,1
1,4111-1111-1111-5624,12795.26,4608.38,42,1
2,4111-1111-1111-5274,12526.10,6764.10,21,3
3,4111-1111-1111-9603,11125.43,7831.18,18,0
4,4111-1111-1111-7629,15876.31,7733.02,36,0
5,4111-1111-1111-1781,15417.23,6878.73,53,0
6,4111-1111-1111-9986,10207.95,998.44,25,4
7,4111-1111-1111-1099,8956.22,6072.88,54,4
8,4111-1111-1111-4583,14386.40,5912.10,35,3
9,4111-1111-1111-9150,16182.95,2487.62,40,2


In [0]:
# MAGIC %md
# MAGIC ### Generating Loan Data


# COMMAND ----------
from pyspark.sql.functions import round, when
from pyspark.sql.types import DecimalType

# 1. Clean up existing table
spark.sql("DROP TABLE IF EXISTS bronze.loans")

# 2. Load users with income data
users = spark.table("bronze.users").select("user_id", "income")

# 3. Generate loan data with business rules
loans_df = users.withColumns({
    # Loan amount = 30-70% of annual income
    "loan_amount": round(
        col("income") * (0.3 + rand() * 0.4), 
        2
    ).cast(DecimalType(10,2)),
    
    # Term based on amount
    "term": when(
        col("loan_amount") < 10000, 12
    ).when(
        col("loan_amount") < 30000, 24
    ).otherwise(36),
    
    # Status probability: 85% active, 10% paid, 5% defaulted
    "status": when(
        rand() < 0.85, "active"
    ).when(
        rand() < 0.95, "paid"
    ).otherwise("defaulted"),
    
    # Interest rate based on risk
    "interest_rate": round(
        5.0 + rand() * 15.0,  # 5-20% range
        2
    )
})

# 4. Data quality assertions
assert loans_df.filter(col("loan_amount").isNull()).count() == 0
assert loans_df.filter(col("loan_amount") > col("income")).count() == 0

# 5. Save with schema enforcement
loans_df.write \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("bronze.loans")

# 6. Verification
print("Loan Data Summary:")
display(loans_df.select(
    "term", "status", "interest_rate"
).summary())

print("\nSample Loans:")
display(loans_df.orderBy("user_id").limit(99))

Loan Data Summary:


summary,term,status,interest_rate
count,100,100,100
mean,31.44,null,11.987
stddev,6.557161248311928,null,4.036244504111427
min,12,active,5.1
25%,24,null,8.8
50%,36,null,11.79
75%,36,null,14.94
max,36,paid,19.99



Sample Loans:


user_id,income,loan_amount,term,status,interest_rate
0,134559,75122.42,36,active,9.07
1,101504,39656.18,36,active,15.96
2,76219,39322.35,36,active,5.72
3,38401,20645.88,24,paid,8.46
4,38393,18327.70,24,paid,12.2
5,106778,49747.85,36,active,12.45
6,147059,49705.86,36,paid,14.97
7,96100,65211.83,36,active,10.13
8,125279,47334.75,36,paid,13.07
9,56835,21443.63,24,active,7.9


In [0]:
# MAGIC %md
# MAGIC ### Silver Layer Transformation

# COMMAND ----------
from pyspark.sql.functions import col, round, when

# 1. Clean up existing silver table
spark.sql("DROP TABLE IF EXISTS silver.credit_profiles")

# 2. Load tables with explicit column selection
users = spark.table("bronze.users").select(
    col("user_id"), 
    col("name"),
    col("age"),
    col("employment_status"),
    col("income").alias("user_income")  # Rename to avoid conflict
)

credit_cards = spark.table("bronze.credit_cards")
loans = spark.table("bronze.loans").select(
    col("user_id"),
    col("loan_amount"),
    col("term"),
    col("status"),
    col("interest_rate")
)

# 3. Join with clear column references
try:
    silver_df = (users
                .join(credit_cards, "user_id", "left")
                .join(loans, "user_id", "left"))
    
    # 4. Calculate metrics using unambiguous columns
    silver_df = silver_df.withColumns({
        "credit_utilization": 
            when(col("limit") > 0, round(col("balance") / col("limit"), 4))
            .otherwise(None),
        
        "dti_ratio": 
            when(col("user_income") > 0, round(col("loan_amount") / col("user_income"), 4))
            .otherwise(None),
        
        "payment_reliability":
            when((col("on_time_payments") + col("late_payments")) > 0,
                 round(col("on_time_payments") / 
                      (col("on_time_payments") + col("late_payments")), 4))
            .otherwise(None)
    })

    # 5. Validate and save
    print(f"Processed records: {silver_df.count()}")
    silver_df.write.mode("overwrite").saveAsTable("silver.credit_profiles")
    display(silver_df.select("user_id", "dti_ratio", "credit_utilization").limit(99))

except Exception as e:
    print(f"Error: {str(e)}")
    raise

Processed records: 100


user_id,dti_ratio,credit_utilization
84,0.5583,0.1343
85,0.3907,0.3602
86,0.5159,0.5400
87,0.5376,0.7039
88,0.4774,0.4871
89,0.4659,0.4462
90,0.3380,0.0978
91,0.6786,0.6781
92,0.3778,0.4110
93,0.3773,0.1537


In [0]:
from pyspark.sql.functions import *

# 1. Clean up existing tables
spark.sql("DROP TABLE IF EXISTS silver.credit_profiles")

# 2. Load and standardize
users = spark.table("bronze.users").select(
    col("user_id").cast("integer").alias("user_id"),
    "name", "age", "employment_status", "income"
)

credit_cards = spark.table("bronze.credit_cards").select(
    col("user_id").cast("integer").alias("user_id"),
    "card_number", "limit", "balance", 
    "on_time_payments", "late_payments"
)

loans = spark.table("bronze.loans").select(
    col("user_id").cast("integer").alias("user_id"),
    "loan_amount", "term", "status", "interest_rate"
)

# 3. Join with validation
try:
    silver_df = users.join(credit_cards, "user_id", "left") \
                   .join(loans, "user_id", "left")
    
    # Verify join integrity
    assert silver_df.count() == users.count(), "Lost records during join!"
    null_users = silver_df.filter(col("user_id").isNull()).count()
    assert null_users == 0, f"Found {null_users} null user_ids"
    
    # 4. Calculate metrics
    silver_df = silver_df.withColumns({
        "credit_utilization": round(col("balance")/col("limit"), 2),
        "dti_ratio": round(col("loan_amount")/col("income"), 4),
        "payment_reliability": round(col("on_time_payments")/(col("on_time_payments")+col("late_payments")), 2)
    })
    
    # 5. Save results
    silver_df.write.mode("overwrite").saveAsTable("silver.credit_profiles")
    print("✅ Silver layer processed successfully!")
    display(silver_df.limit(99))
    
except Exception as e:
    print(f"❌ Failed: {str(e)}")
    raise

✅ Silver layer processed successfully!


user_id,name,age,employment_status,income,card_number,limit,balance,on_time_payments,late_payments,loan_amount,term,status,interest_rate,credit_utilization,dti_ratio,payment_reliability
84,User_84,43,unemployed,32307,4111-1111-1111-3911,14838.27,1992.74,54,1,18036.55,24,active,9.07,0.13,0.5583,0.98
85,User_85,18,unemployed,130851,4111-1111-1111-5624,12795.26,4608.38,42,1,51121.64,36,active,15.96,0.36,0.3907,0.98
86,User_86,67,employed,106622,4111-1111-1111-5274,12526.10,6764.10,21,3,55007.64,36,active,5.72,0.54,0.5159,0.88
87,User_87,25,employed,21253,4111-1111-1111-9603,11125.43,7831.18,18,0,11426.45,24,paid,8.46,0.70,0.5376,1.0
88,User_88,64,employed,98713,4111-1111-1111-7629,15876.31,7733.02,36,0,47122.72,36,paid,12.2,0.49,0.4774,1.0
89,User_89,21,employed,117477,4111-1111-1111-1781,15417.23,6878.73,53,0,54732.51,36,active,12.45,0.45,0.4659,1.0
90,User_90,19,unemployed,72263,4111-1111-1111-9986,10207.95,998.44,25,4,24424.86,24,paid,14.97,0.10,0.3380,0.86
91,User_91,65,employed,128657,4111-1111-1111-1099,8956.22,6072.88,54,4,87304.45,36,active,10.13,0.68,0.6786,0.93
92,User_92,66,unemployed,105071,4111-1111-1111-4583,14386.40,5912.10,35,3,39699.47,36,paid,13.07,0.41,0.3778,0.92
93,User_93,55,unemployed,134732,4111-1111-1111-9150,16182.95,2487.62,40,2,50833.88,36,active,7.9,0.15,0.3773,0.95


In [0]:
# Start with silver data
gold_df = silver_df.withColumn("base_score", lit(650))

gold_df = gold_df.withColumn("dti_penalty", when(col("dti_ratio") > 0.5, lit(30))
                                           .when(col("dti_ratio") > 0.4, lit(20))
                                           .otherwise(lit(0)))

gold_df = gold_df.withColumn("utilization_penalty", when(col("credit_utilization") > 0.7, lit(25))
                                                    .when(col("credit_utilization") > 0.5, lit(15))
                                                    .otherwise(lit(0)))

gold_df = gold_df.withColumn("late_payment_penalty", col("late_payments") * lit(5))

gold_df = gold_df.withColumn("reliability_bonus", when(col("payment_reliability") > 0.95, lit(30))
                                                    .when(col("payment_reliability") > 0.9, lit(15))
                                                    .otherwise(lit(0)))

gold_df = gold_df.withColumn("credit_score", least(greatest(
    col("base_score")
    - col("dti_penalty")
    - col("utilization_penalty")
    - col("late_payment_penalty")
    + col("reliability_bonus"), lit(300)), lit(850)))

gold_df = gold_df.withColumn("risk_category", when(col("credit_score") >= 800, "Excellent")
                                               .when(col("credit_score") >= 740, "Very Good")
                                               .when(col("credit_score") >= 670, "Good")
                                               .when(col("credit_score") >= 580, "Fair")
                                               .otherwise("Poor"))


In [0]:
# Credit score distribution across categories
display(gold_df.groupBy("risk_category").agg(
    count("*").alias("customer_count"),
    avg("credit_score").alias("avg_score"),
    avg("dti_ratio").alias("avg_dti"),
    avg("credit_utilization").alias("avg_utilization"),
    avg("payment_reliability").alias("avg_reliability")
).orderBy("avg_score", ascending=False))


risk_category,customer_count,avg_score,avg_dti,avg_utilization,avg_reliability
Good,5,676.0,0.34946000,0.330000,0.984
Fair,95,626.5263157894736,0.52556632,0.410316,0.9284210526315788


In [0]:
# Top customers by credit score
display(gold_df.select(
    "user_id", 
    "credit_score", 
    "risk_category",
    "dti_ratio", 
    "credit_utilization",
    "late_payments",
    "payment_reliability"
).orderBy("credit_score", ascending=False).limit(20))


user_id,credit_score,risk_category,dti_ratio,credit_utilization,late_payments,payment_reliability
42,680,Good,0.3104,0.22,0,1.0
80,680,Good,0.3254,0.47,0,1.0
85,675,Good,0.3907,0.36,1,0.98
76,675,Good,0.3599,0.45,1,0.98
50,670,Good,0.3609,0.15,2,0.96
96,665,Fair,0.3498,0.68,0,1.0
75,660,Fair,0.3673,0.32,1,0.94
22,660,Fair,0.4315,0.07,0,1.0
88,660,Fair,0.4774,0.49,0,1.0
46,660,Fair,0.4239,0.32,0,1.0


In [0]:
display(gold_df.groupBy("risk_category").agg(
    count("*").alias("customer_count"),
    avg("credit_score").alias("avg_score")
).orderBy("avg_score", ascending=False))

display(gold_df.select("user_id", "credit_score", "risk_category").orderBy("credit_score", ascending=False).limit(10))


risk_category,customer_count,avg_score
Good,5,676.0
Fair,95,626.5263157894736


user_id,credit_score,risk_category
80,680,Good
42,680,Good
85,675,Good
76,675,Good
50,670,Good
96,665,Fair
46,660,Fair
75,660,Fair
22,660,Fair
88,660,Fair


Showing table size


In [0]:
print(f"Total customers processed: {gold_df.count()}")


Total customers processed: 100


Saving snapshot to export

In [0]:
gold_df.write.mode("overwrite").format("delta").save("/tmp/final_credit_scores")


Notebook Finished

## ✅ Project Complete
This notebook demonstrates a full credit scoring pipeline using PySpark and Delta Lake on Databricks Community Edition.
